# Proyecto 1 · Catálogo de experimentos

Este notebook **muestra** la definición de cada experimento. No entrena nada, no necesita GPU y se puede
ejecutar en cualquier momento.

La fuente de verdad sigue siendo `experimentos.py`: cada experimento es una línea `Config(...)` con lo que
cambia respecto a los valores por defecto de `entrenamiento.Config`. Aquí se leen esas definiciones y se
presentan de tres formas:

1. **Tabla general** con todos los experimentos y sus hiperparámetros.
2. **Por parte del enunciado:** qué cambia en cada experimento respecto a su referencia (la comparación controlada).
3. **Ficha de un experimento:** configuración completa, arquitectura del modelo, parámetros y estado.

In [1]:
import dataclasses, json
import pandas as pd
import torch

from config import RESULTADOS
from entrenamiento import Config
from experimentos import EXPERIMENTOS, POR_ID
from modelos import crear_modelo, contar_parametros

pd.set_option("display.max_colwidth", 60, "display.width", 200)
DEFECTO = Config(id="(defecto)")

def a_dict(c):
    d = dataclasses.asdict(c)
    d["modelo_kw"] = str(d["modelo_kw"]) if d["modelo_kw"] else ""
    return d

def estado(id_):
    d = RESULTADOS / id_
    if (d / "resumen.json").exists():
        r = json.loads((d / "resumen.json").read_text())
        return f"terminado · top-1 {r['val_top1']:.1%}"
    if (d / "historial.json").exists():
        h = json.loads((d / "historial.json").read_text())
        return f"en curso · época {len(h)}/{POR_ID[id_].epocas}"
    return "pendiente"

## 1. Tabla general

Una fila por experimento. Las columnas son los campos de `Config`. Las columnas que valen lo mismo en
todos los experimentos (semilla, clip, etc.) se muestran aparte, debajo.

In [2]:
T = pd.DataFrame([a_dict(c) for c in EXPERIMENTOS]).set_index("id")
T.insert(1, "estado", [estado(i) for i in T.index])
constantes = [c for c in T.columns if T[c].nunique() == 1]
print("Iguales en todos los experimentos:", {c: T[c].tolist()[0] for c in constantes})
T.drop(columns=constantes)

Iguales en todos los experimentos: {'epocas': 16, 'warmup': 1.0, 'clip': 5.0, 'paciencia': 4, 'min_delta': 0.001, 'semilla': 42}


,descripcion,estado,modelo,preentrenado,ajuste,bn,dropout,modelo_kw,datos,aumento,muestreo,optimizador,lr,factor_lr_backbone,weight_decay,batch,perdida,label_smoothing
id,,,,,,,,,,,,,,,,,,
E01,MLP 32px (baseline),terminado · top-1 1.1%,mlp,False,completo,True,0.0,"{'res': 32, 'ocultas': (2048, 1024)}",completo,ninguno,uniforme,adam,0.0010,1.0,0.00000,512,ce,0.0
E02,MLP 64px (efecto de la resolución de entrada),terminado · top-1 1.0%,mlp,False,completo,True,0.0,"{'res': 64, 'ocultas': (2048, 1024)}",completo,ninguno,uniforme,adam,0.0010,1.0,0.00000,512,ce,0.0
E03,ResNet-18 desde cero (receta base),terminado · top-1 32.4%,resnet18,False,completo,True,0.0,,completo,basico,uniforme,sgdm,0.1000,1.0,0.00005,256,ce,0.0
E04,ResNet-50 desde cero,terminado · top-1 37.6%,resnet50,False,completo,True,0.0,,completo,basico,uniforme,sgdm,0.1000,1.0,0.00005,256,ce,0.0
E05,EfficientNet-B0 desde cero,terminado · top-1 37.0%,efficientnet_b0,False,completo,True,0.0,,completo,basico,uniforme,sgdm,0.1000,1.0,0.00005,256,ce,0.0
E06,"SGD sin momentum, lr 0.1",terminado · top-1 17.3%,resnet18,False,completo,True,0.0,,completo,basico,uniforme,sgd,0.1000,1.0,0.00005,256,ce,0.0
E07,"SGD+momentum, lr 0.02",terminado · top-1 25.9%,resnet18,False,completo,True,0.0,,completo,basico,uniforme,sgdm,0.0200,1.0,0.00005,256,ce,0.0
E08,"SGD+momentum, lr 0.5",terminado · top-1 34.7%,resnet18,False,completo,True,0.0,,completo,basico,uniforme,sgdm,0.5000,1.0,0.00005,256,ce,0.0
E09,"Adam, lr 1e-3",terminado · top-1 32.6%,resnet18,False,completo,True,0.0,,completo,basico,uniforme,adam,0.0010,1.0,0.00005,256,ce,0.0


## 2. Qué cambia en cada experimento (por parte del enunciado)

Cada experimento se compara con una **referencia**. La función `diferencias` lista solo los campos que
cambian: si un experimento difiere de su referencia en un único campo, el efecto observado se puede
atribuir a ese campo.

In [3]:
def diferencias(id_, ref):
    a, b = a_dict(POR_ID[id_]), a_dict(POR_ID[ref] if ref in POR_ID else DEFECTO)
    return {k: f"{b[k]} → {a[k]}" for k in a if k not in ("id", "descripcion") and a[k] != b[k]}

def tabla_parte(pares):
    filas = [dict(ID=i, referencia=ref, descripción=POR_ID[i].descripcion, estado=estado(i),
                  cambios="; ".join(f"{k}: {v}" for k, v in diferencias(i, ref).items()) or "(es la referencia)")
             for i, ref in pares]
    return pd.DataFrame(filas).set_index("ID")

### Parte 2 · Baseline MLP
Referencia: los valores por defecto. E02 difiere de E01 **solo en la resolución de entrada** (32 → 64 px).

In [4]:
tabla_parte([("E01", "(defecto)"), ("E02", "E01")])

,referencia,descripción,estado,cambios
ID,,,,
E01,(defecto),MLP 32px (baseline),terminado · top-1 1.1%,"modelo: resnet18 → mlp; modelo_kw: → {'res': 32, 'ocult..."
E02,E01,MLP 64px (efecto de la resolución de entrada),terminado · top-1 1.0%,"modelo_kw: {'res': 32, 'ocultas': (2048, 1024)} → {'res'..."


### Parte 3 · Arquitecturas CNN desde cero
Referencia: E03 (receta base). E04 y E05 solo cambian `modelo`: misma receta, mismos datos, mismas épocas.

In [5]:
tabla_parte([("E03", "(defecto)"), ("E04", "E03"), ("E05", "E03")])

,referencia,descripción,estado,cambios
ID,,,,
E03,(defecto),ResNet-18 desde cero (receta base),terminado · top-1 32.4%,(es la referencia)
E04,E03,ResNet-50 desde cero,terminado · top-1 37.6%,modelo: resnet18 → resnet50
E05,E03,EfficientNet-B0 desde cero,terminado · top-1 37.0%,modelo: resnet18 → efficientnet_b0


### Parte 4 · Optimización (ResNet-18)
Referencia: E03 (SGD + momentum, lr 0.1). Cada experimento cambia el optimizador, el learning rate o ambos.

In [6]:
tabla_parte([(i, "E03") for i in ["E06", "E07", "E08", "E09", "E10", "E11", "E12"]])

,referencia,descripción,estado,cambios
ID,,,,
E06,E03,"SGD sin momentum, lr 0.1",terminado · top-1 17.3%,optimizador: sgdm → sgd
E07,E03,"SGD+momentum, lr 0.02",terminado · top-1 25.9%,lr: 0.1 → 0.02
E08,E03,"SGD+momentum, lr 0.5",terminado · top-1 34.7%,lr: 0.1 → 0.5
E09,E03,"Adam, lr 1e-3",terminado · top-1 32.6%,optimizador: sgdm → adam; lr: 0.1 → 0.001
E10,E03,"Adam, lr 1e-4",terminado · top-1 15.6%,optimizador: sgdm → adam; lr: 0.1 → 0.0001
E11,E03,"Adam, lr 1e-2",terminado · top-1 16.7%,optimizador: sgdm → adam; lr: 0.1 → 0.01
E12,E03,"AdamW, lr 1e-3, wd 0.05 (desacoplado)",terminado · top-1 29.3%,optimizador: sgdm → adamw; lr: 0.1 → 0.001; weight_decay...


### Parte 5 · Regularización (ablation study)
Cada experimento se compara con el **anterior**: se añade un componente cada vez.
A6 se compara con A2 para aislar el efecto del aumento básico.

In [7]:
tabla_parte([("A1", "E03"), ("A2", "A1"), ("A3", "A2"), ("A4", "A3"), ("A5", "A4"), ("A6", "A2")])

,referencia,descripción,estado,cambios
ID,,,,
A1,E03,"sin BN, sin dropout, sin aumento, sin wd",terminado · top-1 8.3%,bn: True → False; aumento: basico → ninguno; weight_deca...
A2,A1,+ BatchNorm,terminado · top-1 21.5%,bn: False → True
A3,A2,+ BN + Dropout 0.3,terminado · top-1 21.2%,dropout: 0.0 → 0.3
A4,A3,+ BN + Dropout + aumento fuerte,terminado · top-1 26.5%,aumento: ninguno → fuerte
A5,A4,+ BN + Dropout + aumento fuerte + wd 5e-5,terminado · top-1 29.4%,weight_decay: 0.0 → 5e-05
A6,A2,"BN + aumento básico (sin dropout, sin wd)",terminado · top-1 29.8%,aumento: ninguno → basico


### Parte 6 · Transfer Learning (ResNet-50)
Referencia: E04 (la misma ResNet-50, desde cero). T1→T2→T3 solo cambian `ajuste`, es decir, qué capas se entrenan.

In [8]:
tabla_parte([("T1", "E04"), ("T2", "T1"), ("T3", "T2"), ("T4", "T3")])

,referencia,descripción,estado,cambios
ID,,,,
T1,E04,A: Feature Extraction (backbone congelado),terminado · top-1 26.7%,preentrenado: False → True; ajuste: completo → congelado...
T2,T1,B: Partial Fine-Tuning (layer4 + cabeza),terminado · top-1 43.8%,ajuste: congelado → parcial
T3,T2,C: Full Fine-Tuning,terminado · top-1 50.4%,ajuste: parcial → completo
T4,T3,C + aumento fuerte + label smoothing 0.1 (mejor modelo),terminado · top-1 50.0%,aumento: basico → fuerte; label_smoothing: 0.0 → 0.1


### Parte 7 · Long-tail
Referencia: T3 (misma receta, entrenada con las 500k imágenes balanceadas). L1 cambia solo el tamaño del
conjunto; L2 solo la forma de la distribución; L3–L5 solo la estrategia de mitigación.

In [9]:
tabla_parte([("L1", "T3"), ("L2", "L1"), ("L3", "L2"), ("L4", "L2"), ("L5", "L2")])

,referencia,descripción,estado,cambios
ID,,,,
L1,T3,"balanceado, mismo nº de imágenes que el long-tail (control)",en curso · época 1/16,datos: completo → lt_balanceado
L2,L1,long-tail sin corrección,terminado · top-1 21.9%,datos: lt_balanceado → longtail
L3,L2,long-tail + Weighted Cross-Entropy (1/n),terminado · top-1 19.5%,perdida: ce → ce_ponderada
L4,L2,long-tail + Logit Adjustment (tau=1),terminado · top-1 24.0%,perdida: ce → logit_ajustado
L5,L2,long-tail + Weighted Sampling,terminado · top-1 19.3%,muestreo: uniforme → ponderado


## 3. Ficha de un experimento

Cambia `ID` y vuelve a ejecutar la celda. Muestra:
* la **configuración completa** (lo definido en `experimentos.py` + los valores por defecto);
* la **arquitectura** del modelo tal como la construye `modelos.crear_modelo`, con sus parámetros por bloque
  (y cuáles se entrenan, en los experimentos de transfer learning);
* el **estado** y, si ya terminó, sus métricas finales.

In [10]:
ID = "E01"   # <- cambiar aquí: E01..E12, A1..A6, T1..T4, L1..L5

c = POR_ID[ID]
print(f"{ID} · {c.descripcion} · {estado(ID)}\n")
print("Cambios respecto a los valores por defecto:")
for k, v in diferencias(ID, "(defecto)").items():
    print(f"  {k:20s} {v}")
print("\nConfiguración completa:")
print(json.dumps(dataclasses.asdict(c), indent=2, ensure_ascii=False))

E01 · MLP 32px (baseline) · terminado · top-1 1.1%

Cambios respecto a los valores por defecto:
  modelo               resnet18 → mlp
  modelo_kw             → {'res': 32, 'ocultas': (2048, 1024)}
  aumento              basico → ninguno
  optimizador          sgdm → adam
  lr                   0.1 → 0.001
  weight_decay         5e-05 → 0.0
  batch                256 → 512

Configuración completa:
{
  "id": "E01",
  "descripcion": "MLP 32px (baseline)",
  "modelo": "mlp",
  "preentrenado": false,
  "ajuste": "completo",
  "bn": true,
  "dropout": 0.0,
  "modelo_kw": {
    "res": 32,
    "ocultas": [
      2048,
      1024
    ]
  },
  "datos": "completo",
  "aumento": "ninguno",
  "muestreo": "uniforme",
  "optimizador": "adam",
  "lr": 0.001,
  "factor_lr_backbone": 1.0,
  "weight_decay": 0.0,
  "batch": 512,
  "epocas": 16,
  "warmup": 1.0,
  "clip": 5.0,
  "perdida": "ce",
  "label_smoothing": 0.0,
  "paciencia": 4,
  "min_delta": 0.001,
  "semilla": 42
}


In [11]:
# Arquitectura (se construye en CPU y sin descargar pesos: solo para inspeccionarla)
m = crear_modelo(c.modelo, preentrenado=False, bn=c.bn, dropout=c.dropout, **c.modelo_kw)
if c.modelo != "mlp":
    from modelos import grupos_ajuste
    _, congelados = grupos_ajuste(m, c.modelo, c.ajuste)
    for b in congelados:
        b.requires_grad_(False)

filas = []
for nombre, bloque in m.named_children():
    total = contar_parametros(bloque)
    filas.append(dict(bloque=nombre, tipo=type(bloque).__name__, parámetros=total,
                      entrenables=contar_parametros(bloque, solo_entrenables=True)))
bloques = pd.DataFrame(filas).set_index("bloque")
print(f"Total: {contar_parametros(m) / 1e6:.2f} M parámetros · entrenables: {contar_parametros(m, True) / 1e6:.2f} M")
bloques.style.format({"parámetros": "{:,}", "entrenables": "{:,}"})

Total: 18.65 M parámetros · entrenables: 18.65 M


,tipo,parámetros,entrenables
bloque,,,
cuerpo,Sequential,"8,397,824","8,397,824"
fc,Linear,"10,250,000","10,250,000"


In [12]:
# Detalle capa a capa (para el MLP es corto; para las CNN, largo)
print(m if c.modelo == "mlp" else "\n".join(str(m).splitlines()[:40]) + "\n  ...")

MLP(
  (cuerpo): Sequential(
    (0): Linear(in_features=3072, out_features=2048, bias=True)
    (1): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.0, inplace=False)
    (4): Linear(in_features=2048, out_features=1024, bias=True)
    (5): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): Dropout(p=0.0, inplace=False)
  )
  (fc): Linear(in_features=1024, out_features=10000, bias=True)
)


In [13]:
# Métricas finales, si el experimento ya terminó
f = RESULTADOS / ID / "resumen.json"
if f.exists():
    r = json.loads(f.read_text())
    display(pd.Series({k: v for k, v in r.items() if k not in ("id", "descripcion")}, name=ID).to_frame())
else:
    print(f"{ID}: {estado(ID)}")

,E01
parametros,18647824
parametros_entrenables,18647824
gflops,0.037257
imagenes_train,500000
pasos_por_epoca,976
gpu,NVIDIA RTX A6000
val_loss,8.66573
val_top1,0.01086
val_top5,0.03387
val_f1_macro,0.008389
